In [1]:
import pandas as pd
import plotly.express as px
import pytz
from datetime import datetime

In [2]:
apps_df = pd.read_csv(
    r"C:\Users\ashis\OneDrive\Desktop\GooglePlayProject\Play Store Data.csv"
)

In [3]:
apps_df['Installs'] = apps_df['Installs'].str.replace('+', '')
apps_df['Installs'] = apps_df['Installs'].str.replace(',', '')
apps_df = apps_df[apps_df['Installs'] != 'Free']
apps_df['Installs'] = pd.to_numeric(apps_df['Installs'],errors='coerce')
apps_df = apps_df.dropna(subset=['Installs'])

In [4]:
apps_df['Reviews'] = pd.to_numeric(apps_df['Reviews'],errors='coerce')
apps_df = apps_df.dropna(subset=['Reviews'])

In [5]:
apps_df['Last Updated'] = pd.to_datetime(apps_df['Last Updated'],errors='coerce')
apps_df = apps_df.dropna(subset=['Last Updated'])

In [6]:
apps_df = apps_df[
    (apps_df['Reviews'] > 500) &
    (~apps_df['App'].str.startswith(('x', 'y', 'z'), na=False)) &
    (~apps_df['App'].str.contains('S', na=False)) &
    (apps_df['Category'].str.startswith(('E', 'C', 'B'), na=False))]

In [7]:
apps_df['Category'] = apps_df['Category'].replace({
    'BEAUTY': 'सौंदर्य',
    'BUSINESS': 'வணிகம்',
    'DATING': 'Dating'})

In [8]:
apps_df['Month'] = apps_df['Last Updated'].dt.to_period('M').astype(str)

In [19]:
time_series = apps_df.groupby(
    ['Month', 'Category'])['Installs'].sum().reset_index()

In [20]:
time_series['Growth'] = time_series.groupby('Category')['Installs'].pct_change()

In [23]:
india = pytz.timezone('Asia/Kolkata')
current_time = datetime.now(india)
current_hour = current_time.hour
if 18 <= current_hour < 21:
    fig4 = px.line(
        time_series,
        x='Month',
        y='Installs',
        color='Category',
        title='Installs Trend Over Time by Category',
        markers=True
    )
    growth_data = time_series[
        time_series['Growth'] > 0.20]
    for category in growth_data['Category'].unique():
        category_data = growth_data[
            growth_data['Category'] == category]
        fig4.add_scatter(
            x=category_data['Month'],
            y=category_data['Installs'],
            fill='tozeroy',
            mode='none',
            name=f'{category} Growth Area'
        )
    fig4.update_layout(
        template='plotly_dark',
        width=1200,
        height=700
    )
    fig4.show()
else:
    print("Graph available only between 6 PM IST and 9 PM IST")

Graph available only between 6 PM IST and 9 PM IST
